# PALIMPSEST: Qwen-2.5-Coder-3B CEGIS Training
Optimized for 16GB T4 VRAM, 8-bit AdamW, and Qwen ChatML formatting.

In [1]:
# Install Unsloth and compatible lightweight fine-tuning dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-89f2j12q/unsloth_059f3e3bc02d4ea3baea4f190511358c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-89f2j12q/unsloth_059f3e3bc02d4ea3baea4f190511358c
  Resolved https://github.com/unslothai/unsloth.git to commit 8df1aaf186c4062d486737c414f5f5565b270404
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 11.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 25.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 20.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 26.5 MB/s  0:00:00
  Created wheel for unslot

In [2]:
# Import FastLanguageModel and load 4-bit quantized base model
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [3]:
# Apply parameter-efficient LoRA adapters targeting all linear attention and MLP projections
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.10.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [4]:
# Load dataset and configure Qwen-2.5 ChatML template with ShareGPT mapping
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Configure Qwen-2.5 ChatML template with full custom role/content mapping
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

# Apply chat template formatting across conversations
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

# Load local JSONL dataset and map formatting function
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [5]:
# Initialize 8-bit AdamW SFTTrainer optimized for 16GB T4 GPU VRAM limits
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,                  # Scaled warmup for a longer run
        num_train_epochs=1,               # Replaces max_steps=150 to train on all 15k rows
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,                 # Log every 10 steps to keep the output clean
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/15000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
# Execute fine-tuning loop
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,000 | Num Epochs = 1 | Total steps = 1,875
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.531122
20,2.111384
30,1.251097


Step,Training Loss
10,2.531122
20,2.111384
30,1.251097
40,0.571008


In [ ]:
# Enable native 2x faster inference and run a sample MBA verification prompt
FastLanguageModel.for_inference(model)

# Construct sample input query using Qwen-2.5 ChatML format
messages = [
    {"from": "user", "value": "De-obfuscate this 64-bit MBA expression:\n((((x) ^ (y))) + (((2) * (((x) & (y))))))"},
]

# Format prompt and prepare input tensor
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate completion
outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.batch_decode(outputs)
print(response[0])

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
De-obfuscate this 64-bit MBA expression:
((((x) ^ (y))) + (((2) * (((x) & (y))))))<|im_end|>
<|im_start|>assistant
<think>
AST Node Count: 10. Analyzing data-flow...
</think>
x + y<|im_end|>


In [ ]:
# Export model to 4-bit Medium GGUF format for local Ollama and LangGraph execution
model.save_pretrained_gguf("palimpsest_qwen_3b_cegis", tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in palimpsest_qwen_3b_cegis/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:09<00:00, 34.92s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:30<00:00, 45.03s/it]


Unsloth: Merge process complete. Saved to `/content/palimpsest_qwen_3b_cegis`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11876-mix-ec42614 (app-b11876-mix-ec42614-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['palimpsest_qwen_3b_cegis_gguf/Qwen2.5-Coder-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGU

{'save_directory': 'palimpsest_qwen_3b_cegis',
 'gguf_directory': 'palimpsest_qwen_3b_cegis_gguf',
 'gguf_files': ['palimpsest_qwen_3b_cegis_gguf/Qwen2.5-Coder-3B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'palimpsest_qwen_3b_cegis_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
!pip install -q transformers huggingface_hub


In [ ]:
import shutil
from google.colab import files

# Zip the entire folder containing your GGUF and Modelfile
print("Zipping files... This might take a minute depending on the model size.")
shutil.make_archive("palimpsest_qwen_model", 'zip', "palimpsest_qwen_3b_cegis_gguf")

# Trigger the one-click download
print("Triggering download...")
files.download("palimpsest_qwen_model.zip")

Zipping files... This might take a minute depending on the model size.
Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>